<a href="https://colab.research.google.com/github/CPTR295/NLP-Basic-Using-Pytorch/blob/main/Classification_Surname_Using_Pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Data Pre-Processing

In [ ]:
#Import Dataset
import kagglehub
path = kagglehub.dataset_download('alenic/surname-dataset-classification')
print("Path to dataset files:", path)

100%|██████████| 307k/307k [00:00<00:00, 63.5MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/alenic/surname-dataset-classification/versions/2


In [ ]:
!mv '/root/.cache/kagglehub/datasets/alenic/surname-dataset-classification/versions/2' '/content/'

In [ ]:
import pandas as pd
import numpy as np
import collections
import re
from argparse import Namespace

In [ ]:
args = Namespace(
    raw_dataset_csv = '/content/2/surname-nationality.csv',
    train_proportion = 0.7,
    val_proportion = 0.15,
    test_proportion = 0.15,
    output_munged_csv = '/content/surnames_with_splits.csv',
    seed = 1337
)

In [ ]:
data = pd.read_csv(args.raw_dataset_csv,header=0)

In [ ]:
data.head()

,rank,surname,nationality,frequency,census_count
0,1.0,Tesfaye,Ethiopian,0.011905,1167260
1,2.0,Mohammed,Ethiopian,0.011111,1084839
2,3.0,Getachew,Ethiopian,0.009174,895366
3,4.0,Abebe,Ethiopian,0.008475,825501
4,5.0,Girma,Ethiopian,0.008403,822765


In [ ]:
surnames = data[['surname','nationality']]

In [ ]:

surnames.head()

,surname,nationality
0,Tesfaye,Ethiopian
1,Mohammed,Ethiopian
2,Getachew,Ethiopian
3,Abebe,Ethiopian
4,Girma,Ethiopian


In [ ]:
set(surnames.nationality)

{'Algerian',
 'Arabic',
 'Brazilian',
 'Chilean',
 'Chinese',
 'Czech',
 'Dutch',
 'English',
 'Ethiopian',
 'Finnish',
 'French',
 'German',
 'Greek',
 'Honduran',
 'Indian',
 'Irish',
 'Italian',
 'Japanese',
 'Korean',
 'Malaysian',
 'Mexican',
 'Moroccan',
 'Nepalese',
 'Nicaraguan',
 'Nigerian',
 'Palestinian',
 'Papua New Guinean',
 'Peruvian',
 'Polish',
 'Portuguese',
 'Russian',
 'Scottish',
 'South African',
 'Spanish',
 'Ukrainian',
 'Venezuelan',
 'Vietnamese'}

In [ ]:

#splitting data by nationality
by_nationality  = collections.defaultdict(list)
for _,row in surnames.iterrows():
  by_nationality[row.nationality].append(row.to_dict())

In [ ]:
len(by_nationality['Dutch']) #Example  lenght

1269

In [ ]:
by_nationality.keys()

dict_keys(['Ethiopian', 'Honduran', 'Nigerian', 'Malaysian', 'Chilean', 'Portuguese', 'Papua New Guinean', 'Algerian', 'Brazilian', 'Venezuelan', 'Ukrainian', 'South African', 'Nicaraguan', 'Moroccan', 'Finnish', 'Mexican', 'Palestinian', 'Nepalese', 'Peruvian', 'Dutch', 'Arabic', 'Irish', 'Spanish', 'French', 'German', 'English', 'Korean', 'Indian', 'Vietnamese', 'Scottish', 'Japanese', 'Polish', 'Greek', 'Czech', 'Italian', 'Russian', 'Chinese'])

In [ ]:
#Split the data
final_list=[]
np.random.seed(args.seed)

for _,item_list in sorted(by_nationality.items()):
  np.random.shuffle(item_list)
  n=len(item_list)
  n_train = int(args.train_proportion*n)
  n_val = int(args.val_proportion*n)
  n_test = int(args.test_proportion*n)
  # print(type(item_list))
  # print(type(item_list[0]))
  for item in item_list[:n_train]:
    item['split']='train'
  for item in item_list[n_train:n_train+n_val]:
    item['split']='val'
  for item in item_list[n_train+n_val:]:
    item['split']='test'

  final_list.extend(item_list)

In [ ]:
final_surnames = pd.DataFrame(final_list)

In [ ]:
final_surnames.head()

,surname,nationality,split
0,Aziez,Algerian,train
1,Saadi,Algerian,train
2,Bensalem,Algerian,train
3,Abidat,Algerian,train
4,Belabed,Algerian,train


In [ ]:
final_surnames.split.value_counts()

,count
split,
train,25358
test,5460
val,5423


In [ ]:
final_surnames.to_csv(args.output_munged_csv,index=False)

Classification Surnames With MLP

In [ ]:
from argparse import Namespace
from collections import Counter
import json
import os
import string

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset,DataLoader
from tqdm import tqdm_notebook


Data Vectorization Classes

In [ ]:
#Vocabulary Class
class Vocabulary(object):
  #Class to process test and extract vocabulary for mapping
  def __init__(self,token_to_idx=None,add_unk=True,unk_token="<UNK"):
    #token_to_idx - dict - a map for token to indices
    if token_to_idx is None:
      token_to_idx={}
    self._token_to_idx = token_to_idx

    self._idx_to_token = {idx:token for token,idx in self._token_to_idx.items()}

    self._add_unk = add_unk
    self._unk_token = unk_token

    self.unk_index=-1
    if add_unk:
      self.unk_index = self.add_token(unk_token)

  def to_serializable(self):
    return {'token_to_idx':self._token_to_idx,'add_unk':self._add_unk,'unk_token':self._unk_token}

  @classmethod
  def from_serializable(cls,contents):
    return cls(**contents)

  def add_token(self,token):
    try:
      index=self._token_to_idx[token]
    except KeyError:
      index=len(self._token_to_idx)
      self._token_to_idx[token]=index #Because index starts with zero
      self._idx_to_token[index]=token
    return index

  def add_many(self,tokens):
    return [self.add_token(token) for token in tokens]

  def lookup_token(self,token):
    if self.unk_index >= 0:
        return self._token_to_idx.get(token, self.unk_index)
    else:
        return self._token_to_idx[token]

  def lookup_index(self,index):
    if index not in self._idx_to_token:
      raise KeyError("the index (%d) is not in the Vocabulary" % index)
    return self._idx_to_token[index]

  def __str__(self) -> str:
    return "<Vocabulary(size=%d)>" % len(self)

  def __len__(self):
    return len(self._token_to_idx)

In [ ]:
vocab = Vocabulary(add_unk=False)
vocab.add_many(['a','b','c'])
print(vocab)

<Vocabulary(size=3)>


In [ ]:
#Vecotrizer Class
class SurnameVectorizer(object):
  #Coordinates Vocabularies and put them to use
  def __init__(self,surname_vocab,nationality_vocab):
    self.surname_vocab = surname_vocab
    self.nationality_vocab = nationality_vocab

  def vecotrizer(self,surname):
    vocab = self.surname_vocab
    one_hot = np.zeros(len(vocab),dtype=np.float32)
    for token in surname:
      one_hot[vocab.lookup_token(token)] = 1
    return one_hot

  @classmethod
  def from_dataframe(cls,surname_df):
    surname_vocab = Vocabulary(unk_token="@")
    nationality_vocab = Vocabulary(add_unk=False)
    for index,row in surname_df.iterrows():
      for letter in row.surname:
        surname_vocab.add_token(letter)
      nationality_vocab.add_token(row.nationality)
    return cls(surname_vocab,nationality_vocab)

  @classmethod
  def from_serializable(cls,contents):
    surname_vocab = Vocabulary.from_serializable(contents['surname_vocab'])
    nationality_vocab = Vocabulary.from_serializable(contents['nationality_vocab'])
    return cls(surname_vocab=surname_vocab,nationality_vocab=nationality_vocab)

  def to_serializable(self):
    return {'surname_vocab':self.surname_vocab.to_serializable(),'nationality_vocab':self.nationality_vocab.to_serializable()}

In [ ]:
#Dataset
class SurnameDataset(Dataset):
  def __init__(self,surname_df,vectorizer):
    self.surname_df = surname_df
    self._vectorizer = vectorizer

    self.train_df = self.surname_df[self.surname_df['split']=='train']
    self.train_size = len(self.train_df)

    self.val_df = self.surname_df[self.surname_df['split']=='val']
    self.validation_size = len(self.val_df)

    self.test_df = self.surname_df[self.surname_df['split']=='test']
    self.test_size = len(self.test_df)

    self.lookup_dict = {'train':(self.train_df,self.train_size),
                         'val':(self.val_df,self.validation_size),
                         'test':(self.test_df,self.test_size)}
    self.set_split('train')

    class_counts = surname_df.nationality.value_counts().to_dict()
    def sort_key(item):
      return self._vectorizer.nationality_vocab.lookup_token(item[0])
    sorted_counts = sorted(class_counts.items(),key=sort_key)
    frequencies = [count for _,count in sorted_counts]
    self.class_weights = 1.0 / torch.tensor(frequencies,dtype=torch.float32)

  @classmethod
  def load_dataset_and_make_vectorizer(cls,surname_csv):
    surname_df = pd.read_csv(surname_csv)
    train_surname_df = surname_df[surname_df['split']=='train']
    return cls(surname_df,SurnameVectorizer.from_dataframe(train_surname_df))

  @classmethod
  def load_dataset_and_load_vectorizer(cls,surname_csv,vectorizer_filepath):
    surname_df = pd.read_csv(surname_csv)
    vectorizer = cls.load_vectorizer_only(vectorizer_filepath)
    return cls(surname_df,vectorizer)

  @classmethod
  def load_vectorizer_only(cls,vectorizer_filepath):
    with open(vectorizer_filepath) as fp:
      return SurnameVectorizer.from_serializable(json.load(fp))

  def save_vectorizer(self,vectorizer_filepath):
    with open(vectorizer_filepath,'w') as fp:
      json.dump(self._vectorizer.to_serializable(),fp)

  def get_vectorizer(self):
    return self._vectorizer

  def set_split(self,split='train'):
    self._target_split = split
    self._target_df,self._target_size = self.lookup_dict[split]

  def __getitem__(self, index):
    row = self._target_df.iloc[index]
    surname_vector = self._vectorizer.vecotrizer(row.surname)
    nationality_index = self._vectorizer.nationality_vocab.lookup_token(row.nationality)
    return {'x_surname':surname_vector,'y_nationality':nationality_index}

  def __len__(self):
        return self._target_size

  def get_num_batches(self,batch_size):
    return len(self) // batch_size




In [ ]:
def generator_batches(dataset,batch_size,shuffle=True,drop_last=True,device='cpu'):
  dataloader = DataLoader(dataset=dataset,batch_size=batch_size,shuffle=shuffle,drop_last=drop_last)
  for data_dict in dataloader:
    out_data_dict = {}
    for name,tensor in data_dict.items():
      out_data_dict[name] = data_dict[name].to(device)
    yield out_data_dict

In [ ]:
#The MODEL
class SurnameClassifier(nn.Module):
  #2-layer Multilayer Perceptron for classifying surnames
  def __init__(self,input_dim,hidden_dim,output_dim):
    super(SurnameClassifier,self).__init__()
    self.fc1 = nn.Linear(input_dim,hidden_dim)
    self.fc2 = nn.Linear(hidden_dim,output_dim)

  def forward(self,x_in,apply_softmax=False):
    #x_in - input tensor with (batch,input_dim) shape
    intermediate_vector = F.relu(self.fc1(x_in))
    prediction_vector = self.fc2(intermediate_vector)

    if apply_softmax:
      prediction_vector = F.softmax(prediction_vector,dim=1) #For every row I want most probable char

    return prediction_vector

In [ ]:
def make_train_state(args):
  return {
      'stop_early':False,
      'early_stopping_step':0,
      'early_stopping_best_val':1e8,
      'learning_rate':args.learning_rate,
      'epoch_index':0,
      'train_loss':[],
      'train_acc':[],
      'val_loss':[],
      'val_acc':[],
      'test_loss':-1,
      'test_acc':-1,
      'model_filename':args.model_state_file
       }

In [ ]:
def update_train_state(args,model,train_state):
  if train_state['epoch_index'] == 0:
    torch.save(model.state_dict(),train_state['model_filename'])
    train_state['stop_early'] = False
  elif train_state['epoch_index'] >= 1:
    loss_tm1,loss_t = train_state['val_loss'][-2:]
    if loss_t >= train_state['early_stopping_best_val']:
      train_state['early_stopping_step'] += 1
    else:
      if loss_t < train_state['early_stopping_best_val']:
        torch.save(model.state_dict(),train_state['model_filename'])
      train_state['early_stopping_step'] = 0
    train_state['stop_early'] = train_state['early_stopping_step'] >= args.early_stopping_criteria
  return train_state

In [ ]:
def compute_accuracy(y_pred,y_target):
  _,y_pred_indices = y_pred.max(dim=1)
  n_correct = torch.eq(y_pred_indices,y_target).sum().item()
  return n_correct/len(y_pred_indices)*100.0

In [ ]:
def set_seed_everywhere(seed,cuda):
  np.random.seed(seed)
  torch.manual_seed(seed)
  if cuda:
    torch.cuda.manual_seed_all(seed)

In [ ]:
def handle_dirs(dirpath):
  if not os.path.exists(dirpath):
    os.makedirs(dirpath)

In [ ]:
args = Namespace(
    surname_csv = '/content/surnames_with_splits.csv',
    vectorizer_file = 'vectorizer.json',
    model_state_file = 'model.pth',
    save_dir = '/content/model_storage/',
    hidden_dim=300,
    seed=133,
    num_epochs=100,
    early_stopping_criteria=5,
    learning_rate=0.001,
    batch_size=64,
    cuda=True,
    reload_from_files=False,
    expand_filepaths_to_save_dir=True
)

In [ ]:
if args.expand_filepaths_to_save_dir:
  args.vectorizer_file = os.path.join(args.save_dir,args.vectorizer_file)
  args.model_state_file = os.path.join(args.save_dir,args.model_state_file)
  print("Expanded filepaths: ")
  print("\t{}".format(args.vectorizer_file))
  print("\t{}".format(args.model_state_file))

Expanded filepaths: 
	/content/model_storage/vectorizer.json
	/content/model_storage/model.pth


In [ ]:
if not torch.cuda.is_available():
  args.cuda=False
args.device = torch.device("cuda" if args.cuda else "cpu")
print(args.device)


cuda


In [ ]:
set_seed_everywhere(args.seed, args.cuda)
handle_dirs(args.save_dir)

In [ ]:
if args.reload_from_files:
  print('Reloading')
  dataset = SurnameDataset.load_dataset_and_load_vectorizer(args.surname_csv,args.vectorizer_file)
else:
  print('Creating')
  dataset = SurnameDataset.load_dataset_and_make_vectorizer(args.surname_csv)
  dataset.save_vectorizer(args.vectorizer_file)

vectorizer = dataset.get_vectorizer()
classifier = SurnameClassifier(input_dim=len(vectorizer.surname_vocab),hidden_dim=args.hidden_dim,output_dim=len(vectorizer.nationality_vocab))

Creating


In [ ]:
classifier = classifier.to(args.device)
dataset.class_weights = dataset.class_weights.to(args.device)

loss_func = nn.CrossEntropyLoss(weight=dataset.class_weights)
optimizer = optim.Adam(classifier.parameters(), lr=args.learning_rate)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer=optimizer,
    mode='min',
    patience=1
)

In [ ]:
train_state = make_train_state(args)

In [ ]:
try:
    for epoch_index in range(args.num_epochs):
        train_state['epoch_index'] = epoch_index

        # =======================
        # Training
        # =======================
        dataset.set_split('train')
        batch_generator = generator_batches(
            dataset,
            batch_size=args.batch_size,
            device=args.device
        )

        classifier.train()
        running_loss = 0.0
        running_acc = 0.0

        for batch_index, batch_dict in enumerate(batch_generator, start=1):
            optimizer.zero_grad()

            y_pred = classifier(x_in=batch_dict['x_surname'])

            loss = loss_func(y_pred, batch_dict['y_nationality'])
            loss.backward()
            optimizer.step()

            running_loss += (loss.item() - running_loss) / batch_index

            acc = compute_accuracy(y_pred, batch_dict['y_nationality'])
            running_acc += (acc - running_acc) / batch_index

        train_state['train_loss'].append(running_loss)
        train_state['train_acc'].append(running_acc)

        # =======================
        # Validation
        # =======================
        dataset.set_split('val')
        batch_generator = generator_batches(
            dataset,
            batch_size=args.batch_size,
            device=args.device
        )

        classifier.eval()
        running_loss = 0.0
        running_acc = 0.0

        with torch.no_grad():
            for batch_index, batch_dict in enumerate(batch_generator, start=1):

                y_pred = classifier(x_in=batch_dict['x_surname'])

                loss = loss_func(y_pred, batch_dict['y_nationality'])

                running_loss += (loss.item() - running_loss) / batch_index

                acc = compute_accuracy(y_pred, batch_dict['y_nationality'])
                running_acc += (acc - running_acc) / batch_index

        train_state['val_loss'].append(running_loss)
        train_state['val_acc'].append(running_acc)

        train_state = update_train_state(
            args=args,
            model=classifier,
            train_state=train_state
        )

        scheduler.step(running_loss)

        print(
            f"Epoch [{epoch_index+1}/{args.num_epochs}] | "
            f"Train Loss: {train_state['train_loss'][-1]:.4f} | "
            f"Train Acc: {train_state['train_acc'][-1]:.2f}% | "
            f"Val Loss: {train_state['val_loss'][-1]:.4f} | "
            f"Val Acc: {train_state['val_acc'][-1]:.2f}%"
        )

        if train_state['stop_early']:
            print("Early stopping triggered.")
            break

except KeyboardInterrupt:
    print("Training interrupted.")

Epoch [1/100] | Train Loss: 3.1506 | Train Acc: 26.28% | Val Loss: 2.8061 | Val Acc: 26.15%
Epoch [2/100] | Train Loss: 2.6516 | Train Acc: 29.67% | Val Loss: 2.6341 | Val Acc: 29.97%
Epoch [3/100] | Train Loss: 2.5205 | Train Acc: 30.51% | Val Loss: 2.5770 | Val Acc: 30.77%
Epoch [4/100] | Train Loss: 2.4575 | Train Acc: 31.00% | Val Loss: 2.5374 | Val Acc: 31.42%
Epoch [5/100] | Train Loss: 2.4061 | Train Acc: 31.87% | Val Loss: 2.5167 | Val Acc: 29.95%
Epoch [6/100] | Train Loss: 2.3597 | Train Acc: 32.31% | Val Loss: 2.4898 | Val Acc: 32.40%
Epoch [7/100] | Train Loss: 2.3260 | Train Acc: 32.99% | Val Loss: 2.4828 | Val Acc: 30.36%
Epoch [8/100] | Train Loss: 2.2875 | Train Acc: 33.51% | Val Loss: 2.4662 | Val Acc: 29.50%
Epoch [9/100] | Train Loss: 2.2574 | Train Acc: 34.23% | Val Loss: 2.4559 | Val Acc: 31.68%
Epoch [10/100] | Train Loss: 2.2208 | Train Acc: 34.69% | Val Loss: 2.4480 | Val Acc: 32.57%
Epoch [11/100] | Train Loss: 2.1910 | Train Acc: 35.30% | Val Loss: 2.4491 | Va

In [ ]:
classifier = classifier.to(args.device)
dataset.class_weights = dataset.class_weights.to(args.device)

loss_func = nn.CrossEntropyLoss(dataset.class_weights)
optimizer = optim.Adam(classifier.parameters(),lr=args.learning_rate)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimizer,mode='min',patience=1) #To adjust Learing Rate

train_state = make_train_state(args)

epoch_bar = tqdm_notebook(desc='training routine',total=args.num_epochs,position=0)

dataset.set_split('train')
train_bar = tqdm_notebook(desc='split=train',total=dataset.get_num_batches(args.batch_size),position=1,leave=False)

dataset.set_split('val')
val_bar = tqdm_notebook(desc='split=val',total=dataset.get_num_batches(args.batch_size),position=1,leave=False)

try:
  for epoch_index in range(args.num_epochs):
    train_state['epoch_index'] = epoch_index
    dataset.set_split('train')
    batch_generator = generator_batches(dataset,batch_size=args.batch_size,device=args.device)
    running_loss = 0.0
    running_acc = 0.0
    classifier.train()
    for batch_index,batch_dict in enumerate(batch_generator):
      optimizer.zero_grad()
      y_pred = classifier(x_in=batch_dict['x_surname'])
      loss = loss_func(y_pred,batch_dict['y_nationality'])
      loss_t = loss.item()
      running_loss += (loss_t - running_loss) / (batch_index + 1)
      loss.backward()
      optimizer.step()
      acc_t = compute_accuracy(y_pred,batch_dict['y_nationality'])
      running_acc += (acc_t - running_acc) / (batch_index +1)
      train_bar.set_postfix(loss=running_loss,acc=running_acc,epoch=epoch_index)
      train_bar.update()
    train_state['train_loss'].append(running_loss)
    train_state['train_acc'].append(running_acc)
    dataset.set_split('val')
    batch_generator = generator_batches(dataset,batch_size=args.batch_size,device=args.device)
    running_loss = 0.0
    running_acc = 0.0
    classifier.eval()
    for batch_index,batch_dict in enumerate(batch_generator):
      y_pred = classifier(x_in=batch_dict['x_surname'])
      loss = loss_func(y_pred,batch_dict['y_nationality'])
      loss_t = loss.item()
      running_loss += (loss_t - running_loss) / (batch_index + 1)
      acc_t = compute_accuracy(y_pred,batch_dict['y_nationality'])
      running_acc += (acc_t - running_acc) / (batch_index +1)
      val_bar.set_postfix(loss=running_loss,acc=running_acc,epoch=epoch_index)
      val_bar.update()

    train_state['val_loss'].append(running_loss)
    train_state['val_acc'].append(running_acc)
    train_state = update_train_state(args=args,model=classifier,train_state=train_state)

    scheduler.step(train_state['val_loss'][-1])
    if train_state['stop_early']:
      break
    train_bar.n = 0
    val_bar.n = 0
    epoch_bar.update()
except KeyboardInterrupt:
  print('Exiting Loop')

/tmp/ipykernel_476/3963110890.py:10: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  epoch_bar = tqdm_notebook(desc='training routine',total=args.num_epochs,position=0)


training routine:   0%|          | 0/100 [00:00<?, ?it/s]

/tmp/ipykernel_476/3963110890.py:13: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  train_bar = tqdm_notebook(desc='split=train',total=dataset.get_num_batches(args.batch_size),position=1,leave=False)


split=train:   0%|          | 0/396 [00:00<?, ?it/s]

/tmp/ipykernel_476/3963110890.py:16: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  val_bar = tqdm_notebook(desc='split=val',total=dataset.get_num_batches(args.batch_size),position=1,leave=False)


split=val:   0%|          | 0/84 [00:00<?, ?it/s]

In [ ]:
classifier.load_state_dict(torch.load(train_state['model_filename']))
classifier = classifier.to(args.device)
dataset.class_weights = dataset.class_weights.to(args.device)
loss_func = nn.CrossEntropyLoss(dataset.class_weights)

dataset.set_split('test')
batch_generator = generator_batches(dataset,batch_size=args.batch_size,device=args.device)
running_loss = 0.0
running_acc = 0.0
classifier.eval()
for batch_index,batch_dict in enumerate(batch_generator):
  y_pred = classifier(x_in=batch_dict['x_surname'])
  loss = loss_func(y_pred,batch_dict['y_nationality'])
  loss_t = loss.item()
  running_loss += (loss_t - running_loss) / (batch_index + 1)
  acc_t = compute_accuracy(y_pred,batch_dict['y_nationality'])
  running_acc += (acc_t - running_acc) / (batch_index +1)
train_state['test_loss'] = running_loss
train_state['test_acc'] = running_acc

In [ ]:
print("Test loss: {:.3f}".format(train_state['test_loss']))
print("Test Accuracy: {:.2f}".format(train_state['test_acc']))

Test loss: 2.417
Test Accuracy: 34.39


In [ ]:
def predict_nationality(surname,classifier, vectorizer):
  vectorized_surname = vectorizer.vecotrizer(surname)
  vectorized_surname = torch.tensor(vectorized_surname).view(1, -1)
  result = classifier(vectorized_surname, apply_softmax=True)

  probability_values, indices = result.max(dim=1)
  index = indices.item()

  predicted_nationality = vectorizer.nationality_vocab.lookup_index(index)
  probability_value = probability_values.item()

  return {'nationality': predicted_nationality, 'probability': probability_value}

In [ ]:
new_surname = input("Enter a surname to classify: ")
classifier = classifier.to("cpu")
prediction = predict_nationality(new_surname, classifier, vectorizer)
print("{} -> {} (p={:0.2f})".format(new_surname,
                                    prediction['nationality'],
                                    prediction['probability']))

Enter a surname to classify: Roy
Roy -> Korean (p=0.11)
